# <p><center style="font-family:newtimeroman;font-size:180%;">Generative AI Chatbot for Alzheimer Disease</center></p>
# <p><center style="font-family:newtimeroman;font-size:180%;">Leader</center></p>
## Steps:

| Step |Description                                     |
|------|------------------------------------------------|
|0|[Introduction](#0)|
| 1    |[Import Libraries](#1)                          |
| 2    |[Read Data](#2)                                 |    
|3   |[Data Spliting](#3)|
|4|[Helper Function for GPT-2](#4)|
|5|[Training GPT-2](#5)|
|6|[Evaluation GPT-2 ](#6)|
|7|[Generation Using Custom GPT-2](#7)

**<a id="0"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Introduction </p>
<a class="btn" href="#home">Tabel of Contents</a>

#### Generative AI Chatbot for Alzheimer Disease
This notebook builds PreTrained Transformer GPT-2 in TensorFlow.

It serves to understand how each part of the Transformer works and how they all fit together.

The Transformer is then tested on a simple seq2seq task : Generative AI Chatbot for Alzheimer Disease.

**<a id="1"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Import Libraries </p>
<a class="btn" href="#home">Tabel of Contents</a>

In [1]:
from transformers import (TextDataset, DataCollatorForLanguageModeling,GPT2Tokenizer,
                          GPT2LMHeadModel,Trainer, TrainingArguments)
import pandas as pd
import matplotlib.pyplot as plt
import wandb
wandb.init(mode="disabled")
import warnings
warnings.filterwarnings('ignore')

2024-03-11 12:00:41.813767: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-11 12:00:41.813870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-11 12:00:41.971211: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


**<a id="2"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Read Data </p>
<a class="btn" href="#home">Tabel of Contents</a>

In [2]:
data=pd.read_csv('/kaggle/input/alzhimer-chat-leader/full_Chat_data.csv',usecols=[1,2])
data.sample(5)

,Questions,Answers
8892,What should individuals do if they notice chan...,Individuals who notice changes in their cognit...
9132,Can Alzheimer's disease following head trauma ...,"Yes, Alzheimer's disease following head trauma..."
25152,How can Alzheimer's risk be reduced?,"A healthy lifestyle, including mental and phys..."
15579,Are there community resources available to sup...,"Yes, there are community resources available t..."
21282,How does evaluating the patient's financial ma...,Assessing financial management skills helps id...


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25177 entries, 0 to 25176
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Questions  25173 non-null  object
 1   Answers    25154 non-null  object
dtypes: object(2)
memory usage: 393.5+ KB


**<a id="3"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Data Spliting </p>
<a class="btn" href="#home">Tabel of Contents</a>

In [4]:
data.to_csv('/kaggle/working/train.csv')

**<a id="4"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Helper Function for GPT-2 </p>
<a class="btn" href="#home">Tabel of Contents</a>

In [5]:
def load_dataset(file_path, tokenizer, block_size = 1024):
    dataset_train = TextDataset(
        tokenizer = tokenizer,
        file_path = file_path,
        block_size = block_size,
    )
    return dataset_train

In [6]:
def load_data_collator(tokenizer, mlm = False):
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, 
        mlm=mlm,
    )
    return data_collator

In [7]:
def train(train_file_path, model_name, output_dir, overwrite_output_dir,
          per_device_train_batch_size, num_train_epochs):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    # Load datasets
    train_dataset = load_dataset(train_file_path, tokenizer)

    # Load data collator
    data_collator = load_data_collator(tokenizer)

    # Save tokenizer
    tokenizer.save_pretrained(output_dir)

    # Load or initialize model
    model = GPT2LMHeadModel.from_pretrained(model_name)

    # Save model
    model.save_pretrained(output_dir)

    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=overwrite_output_dir,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        logging_dir="./logs",
        logging_steps=100,  # Log every 100 steps
        save_steps=500,  # Save checkpoint every 500 steps
        logging_first_step=True,
        save_total_limit=2,
        learning_rate=.0001
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )
    hist = trainer.train()
    trainer.save_model()
    return trainer,hist

**<a id="5"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Training GPT-2 </p>
<a class="btn" href="#home">Tabel of Contents</a>

In [8]:
train_file_path = "/kaggle/working/train.csv"
model_name = 'gpt2'
output_dir = '/kaggle/working/custom_model'
overwrite_output_dir = True
per_device_train_batch_size = 4
num_train_epochs = 50

In [9]:
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}

In [10]:
model = GPT2LMHeadModel.from_pretrained(model_name)
model

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [11]:
model.base_model

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D()
        (c_proj): Conv1D()
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D()
        (c_proj): Conv1D()
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [12]:
# Train
hist=train(
    train_file_path=train_file_path,
    model_name=model_name,
    output_dir=output_dir,
    overwrite_output_dir=overwrite_output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
)    

Step,Training Loss
1,2.052200
100,1.683500
200,1.476700
300,1.413500
400,1.317200
500,1.225500
600,1.189800
700,1.154500
800,1.133500
900,1.069700


**<a id="6"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Evaluation GPT-2 </p>
<a class="btn" href="#home">Tabel of Contents</a>

In [13]:
print("Global Step:", hist[1].global_step)
print("Epoch:", hist[1].metrics['epoch'])
print("Train Runtime:", hist[1].metrics['train_runtime'])
print("Train Samples Per Second:", hist[1].metrics['train_samples_per_second'])
print("Train Steps Per Second:", hist[1].metrics['train_steps_per_second'])
print("Total FLOPS:", hist[1].metrics['total_flos'])
print("Train Loss:", hist[1].metrics['train_loss'])

Global Step: 20550
Epoch: 50.0
Train Runtime: 14786.4876
Train Samples Per Second: 5.549
Train Steps Per Second: 1.39
Total FLOPS: 4.28780224512e+16
Train Loss: 0.3342023765257675


**<a id="7"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:120%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Generation Using Custom GPT-2 </p>
<a class="btn" href="#home">Tabel of Contents</a>

In [14]:
def load_model(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    return model


def load_tokenizer(tokenizer_path):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_path)
    return tokenizer

def generate_text(model_path, sequence, max_length):
    
    model = load_model(model_path)
    tokenizer = load_tokenizer(model_path)
    ids = tokenizer.encode(f'{sequence}', return_tensors='pt')
    final_outputs = model.generate(
        ids,
        do_sample=True,
        max_length=max_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )
    return tokenizer.decode(final_outputs[0], skip_special_tokens=True)

In [15]:
model_path = "/kaggle/working/custom_model"
sequence = data['Questions'].iloc[100]
max_len = 100
print("Q : ",data['Questions'].iloc[100])
print()
print("A : ",data['Answers'].iloc[100])
print()
print("G : ",generate_text(model_path, sequence, max_len)) 

Q :  Are there lifestyle changes that can positively impact cognitive health?

A :  Engaging in mentally and socially stimulating activities, such as social events, reading, playing board games, and playing an instrument, is linked to preserved thinking skills and a lower risk of Alzheimer's. Lifestyle interventions, including diet, exercise, and social activities, have shown positive results in reducing cognitive decline among those at risk.

G :  Are there lifestyle changes that can positively impact cognitive health?,"Engaging in mentally stimulating activities, maintaining a healthy diet rich in antioxidants and omega-3 fatty acids, staying physically and socially active, and managing cardiovascular risk factors can positively impact cognitive health. These lifestyle factors are associated with better cognitive performance and may help lower the risk of Alzheimer's disease."
6879,"What is the relationship between hormonal changes in postmenopausal women and Alzheimer's risk, and ar

In [16]:
sequence = data['Questions'].iloc[50]
max_len = 200
print("Q : ",data['Questions'].iloc[50])
print()
print("A : ",data['Answers'].iloc[50])
print()
print("G : ",generate_text(model_path, sequence, max_len)) 

Q :  What are some daily activities that people with Alzheimer's may enjoy?

A :  People with Alzheimer's can engage in various daily activities to keep their days interesting and fun. These activities include household chores like washing dishes or sorting mail, cooking and baking, exercise such as walking or using a stationary bike, enjoying music and dancing, spending time with pets, gardening, and visiting with children. Adapting these activities to their abilities ensures a positive and engaging experience.

G :  What are some daily activities that people with Alzheimer's may enjoy?,"People with Alzheimer's can engage in various daily activities to keep their days interesting and fun. These activities include household chores, cooking videos, exercise, listening to music, and spending time with pets. For adults, these activities can be therapeutic, providing opportunities for self-expression and enjoyment."
16765,How can family members involve someone with Alzheimer's in holiday c

In [17]:
sequence = data['Questions'].iloc[150]
max_len = 200
print("Q : ",data['Questions'].iloc[150])
print()
print("A : ",data['Answers'].iloc[150])
print()
print("G : ",generate_text(model_path, sequence, max_len)) 

Q :   Could you explain what causes Alzheimer's disease and how it progresses?

A :  Alzheimer's disease is quite complex, involving the build-up of substances like amyloid and tau in the brain. These form plaques and tangles, disrupting brain function. This process leads to brain shrinkage and difficulties in memory and thinking, eventually reaching a stage referred to as 'dementia.'

G :   Could you explain what causes Alzheimer's disease and how it progresses?,"Alzheimer's disease is quite complex, involving the build-up of substances called amyloid and tau in the brain. These form plaques and tangles, disrupting communication between brain cells and causing cell death..."
17578,Are there any early warning signs of Alzheimer's disease?,"Yes, early signs of Alzheimer's disease may include memory loss, difficulty with problem-solving, confusion, and changes in mood or behavior. These signs may not significantly interfere with daily functioning but are valuable for early detection..."


In [18]:
sequence = "what is alzheimer's disease"
max_len = 500
print("Q : ",sequence)
print()
print("G : ",generate_text(model_path, sequence, max_len)) 

Q :  what is alzheimer's disease

G :  what is alzheimer's disease?,"Alzheimer’s disease is a neurological condition that causes a decline in thinking skills, memory, and the ability to perform everyday tasks. It is the most common form of dementia, accounting for 60-80% of cases globally."
16879,What are the key symptoms of Alzheimer’s disease?,"Early signs of Alzheimer’s disease include memory loss, difficulty in problem-solving, confusion about time or place, challenges in completing familiar tasks, and changes in mood or personality."
16880,How is Alzheimer’s disease diagnosed?,"Diagnosis involves a thorough medical history, physical and neurological exams, cognitive assessments, and sometimes brain imaging. Definitive diagnosis is usually post-mortem by examining the brain tissue."
16881,Can Alzheimer’s disease be prevented or cured?,"Currently, there is no cure for Alzheimer’s disease. Available treatments focus on managing symptoms and may include medications to improve cognitiv